In [5]:
import torch
import pandas as pd
import numpy as np
import transformers
import peft

from pathlib import Path
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed
from peft import PeftModel
from datasets import load_from_disk

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

models_dir = project_dir / "models"
reports_dir = project_dir / "reports"
data_dir = project_dir / "data"

BASE_MODEL = "google/flan-t5-small"
ADAPTER_PATH = models_dir / "healthbrief_best_adapter"

SEED = 42
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL
)

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False
)

model.to(device)
model.eval()

print("Model Type:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("Training Mode:", model.training)

model.print_trainable_parameters()

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4384.40it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model Type: PeftModelForSeq2SeqLM
Device: cuda:0
Training Mode: False
trainable params: 0 || all params: 77,305,216 || trainable%: 0.0000


In [8]:
environment_df = pd.read_csv(
    reports_dir / "training_environment.csv"
)

display(environment_df.T)

base_config = model.get_base_model().config

print("Transformers Version:", transformers.__version__)
print("PEFT Version:", peft.__version__)
print("Model Revision:", getattr(base_config, "_commit_hash", None))
print("Model Type:", base_config.model_type)
print("Tie Word Embeddings:", base_config.tie_word_embeddings)
print("Model Dimension:", base_config.d_model)
print("Vocabulary Size:", base_config.vocab_size)
print("Model Precision:", next(model.parameters()).dtype)
print("Training Mode:", model.training)

model.print_trainable_parameters()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
feature,python_version,pytorch_version,transformers_version,peft_version,datasets_version,cuda_version,cuda_available,gpu_name,seed,dataset_name,dataset_revision,model_name,model_revision,tokenizer_revision
value,3.14.4,2.14.1+cu130,5.14.1,0.22.0,5.1.0,13.0,True,NVIDIA GeForce RTX 4060 Laptop GPU,42,BioLaySumm/BioLaySumm2025-eLife,144e9c785e3a309da804eb6786feb38e6596f390,google/flan-t5-small,0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab,0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab


Transformers Version: 5.14.1
PEFT Version: 0.22.0
Model Revision: 0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab
Model Type: t5
Tie Word Embeddings: True
Model Dimension: 512
Vocabulary Size: 32128
Model Precision: torch.float32
Training Mode: False
trainable params: 0 || all params: 77,305,216 || trainable%: 0.0000


In [9]:
environment_df = pd.read_csv(
    reports_dir / "training_environment.csv"
)

display(environment_df.T)

base_config = model.get_base_model().config

print("Transformers Version:", transformers.__version__)
print("PEFT Version:", peft.__version__)
print("Model Revision:", getattr(base_config, "_commit_hash", None))
print("Model Type:", base_config.model_type)
print("Tie Word Embeddings:", base_config.tie_word_embeddings)
print("Model Dimension:", base_config.d_model)
print("Vocabulary Size:", base_config.vocab_size)
print("Model Precision:", next(model.parameters()).dtype)
print("Training Mode:", model.training)

model.print_trainable_parameters()

input_weights = model.get_input_embeddings().weight
output_weights = model.get_output_embeddings().weight

weights_tied = (
    input_weights.data_ptr() == output_weights.data_ptr()
)

weights_equal = torch.equal(
    input_weights,
    output_weights
)

print("Input Weight Shape:", input_weights.shape)
print("Output Weight Shape:", output_weights.shape)
print("Weights Tied:", weights_tied)
print("Weights Equal:", weights_equal)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
feature,python_version,pytorch_version,transformers_version,peft_version,datasets_version,cuda_version,cuda_available,gpu_name,seed,dataset_name,dataset_revision,model_name,model_revision,tokenizer_revision
value,3.14.4,2.14.1+cu130,5.14.1,0.22.0,5.1.0,13.0,True,NVIDIA GeForce RTX 4060 Laptop GPU,42,BioLaySumm/BioLaySumm2025-eLife,144e9c785e3a309da804eb6786feb38e6596f390,google/flan-t5-small,0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab,0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab


Transformers Version: 5.14.1
PEFT Version: 0.22.0
Model Revision: 0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab
Model Type: t5
Tie Word Embeddings: True
Model Dimension: 512
Vocabulary Size: 32128
Model Precision: torch.float32
Training Mode: False
trainable params: 0 || all params: 77,305,216 || trainable%: 0.0000
Input Weight Shape: torch.Size([32128, 512])
Output Weight Shape: torch.Size([32128, 512])
Weights Tied: False
Weights Equal: False


In [10]:
tokenized_dataset = load_from_disk(
    str(data_dir / "healthbrief_tokenized")
)

previous_results = pd.read_csv(
    reports_dir / "qualitative_summary_comparison.csv"
)

VALIDATION_INDEX = 7

previous_sample = previous_results[
    (previous_results["validation_index"] == VALIDATION_INDEX)
    & (previous_results["model"] == "lora")
].iloc[0]

previous_summary = previous_sample["generated_summary"]
previous_token_count = int(previous_sample["generated_tokens"])

print("Validation Index:", VALIDATION_INDEX)
print("Previous Token Count:", previous_token_count)
print("Previous Summary:", previous_summary[:300])

Validation Index: 7
Previous Token Count: 767
Previous Summary: The fusion proteins are a fusion of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule o


In [11]:
GENERATION_MAX_LENGTH = 768
GENERATION_NUM_BEAMS = 1
DO_SAMPLE = False

model.generation_config.max_length = GENERATION_MAX_LENGTH
model.generation_config.max_new_tokens = None
model.generation_config.num_beams = GENERATION_NUM_BEAMS
model.generation_config.do_sample = DO_SAMPLE

print(model.generation_config)

GenerationConfig {
  "decoder_start_token_id": 0,
  "do_sample": false,
  "eos_token_id": 1,
  "max_length": 768,
  "num_beams": 1,
  "pad_token_id": 0
}



In [12]:
from contextlib import nullcontext

batch_indices = [7, 8, 12, 17]

inputs = tokenizer.pad(
    [
        {
            "input_ids": tokenized_dataset["validation"][i]["input_ids"],
            "attention_mask": tokenized_dataset["validation"][i]["attention_mask"]
        }
        for i in batch_indices
    ],
    padding=True,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

precision_context = (
    torch.autocast("cuda", dtype=torch.bfloat16)
    if device == "cuda"
    else nullcontext()
)

model.eval()

with torch.inference_mode(), precision_context:

    generated = model.generate(
        **inputs,
        max_length=GENERATION_MAX_LENGTH,
        num_beams=GENERATION_NUM_BEAMS,
        do_sample=DO_SAMPLE
    )

generated_ids = generated[0].cpu()

new_summary = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)

special_token_ids = set(tokenizer.all_special_ids)

new_token_count = sum(
    token_id not in special_token_ids
    for token_id in generated_ids.tolist()
)

print("Generated Token Count:", new_token_count)
print("Generated Summary:", new_summary[:300])

Generated Token Count: 767
Generated Summary: The fusion proteins are a fusion of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule of a molecule o


In [13]:
summary_match = previous_summary == new_summary

token_count_match = previous_token_count == new_token_count

eos_generated = tokenizer.eos_token_id in generated_ids[1:].tolist()

reload_comparison = pd.DataFrame([{
    "validation_index": VALIDATION_INDEX,
    "previous_token_count": previous_token_count,
    "reloaded_token_count": new_token_count,
    "token_count_match": token_count_match,
    "summary_match": summary_match,
    "eos_generated": eos_generated,
    "previous_summary": previous_summary,
    "reloaded_summary": new_summary
}])

display(reload_comparison)

print("Summary Match:", summary_match)
print("Token Count Match:", token_count_match)
print("EOS Generated:", eos_generated)

,validation_index,previous_token_count,reloaded_token_count,token_count_match,summary_match,eos_generated,previous_summary,reloaded_summary
0,7,767,767,True,True,False,The fusion proteins are a fusion of a molecule...,The fusion proteins are a fusion of a molecule...


Summary Match: True
Token Count Match: True
EOS Generated: False


In [14]:
reload_comparison["transformers_version"] = transformers.__version__
reload_comparison["peft_version"] = peft.__version__

reload_comparison["base_model"] = BASE_MODEL
reload_comparison["base_model_revision"] = getattr(
    base_config, "_commit_hash", None
)

reload_comparison["tie_word_embeddings_config"] = (
    base_config.tie_word_embeddings
)

reload_comparison["weights_tied"] = weights_tied
reload_comparison["weights_equal"] = weights_equal

reload_comparison["model_dtype"] = str(
    next(model.parameters()).dtype
)

reload_comparison["inference_precision"] = (
    "bf16_autocast" if device == "cuda" else "float32"
)

reload_comparison["generation_max_length"] = GENERATION_MAX_LENGTH
reload_comparison["generation_num_beams"] = GENERATION_NUM_BEAMS
reload_comparison["do_sample"] = DO_SAMPLE

reload_comparison["trainable_parameters"] = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

reload_comparison["reload_verification_pass"] = (
    summary_match and token_count_match
)

reload_comparison.to_csv(
    reports_dir / "adapter_reload_verification.csv",
    index=False
)

display(reload_comparison.T)

,0
validation_index,7
previous_token_count,767
reloaded_token_count,767
token_count_match,True
summary_match,True
eos_generated,False
previous_summary,The fusion proteins are a fusion of a molecule...
reloaded_summary,The fusion proteins are a fusion of a molecule...
transformers_version,5.14.1
peft_version,0.22.0


In [15]:
generation_config = model.generation_config

generation_settings = {
    "max_length": generation_config.max_length,
    "max_new_tokens": generation_config.max_new_tokens,
    "num_beams": generation_config.num_beams,
    "do_sample": generation_config.do_sample,
    "repetition_penalty": generation_config.repetition_penalty,
    "no_repeat_ngram_size": generation_config.no_repeat_ngram_size,
    "eos_token_id": generation_config.eos_token_id,
    "forced_eos_token_id": generation_config.forced_eos_token_id,
    "decoder_start_token_id": generation_config.decoder_start_token_id,
    "pad_token_id": generation_config.pad_token_id
}

display(pd.DataFrame(
    generation_settings.items(),
    columns=["parameter", "value"]
))

,parameter,value
0,max_length,768
1,max_new_tokens,None
2,num_beams,1
3,do_sample,False
4,repetition_penalty,None
5,no_repeat_ngram_size,None
6,eos_token_id,1
7,forced_eos_token_id,None
8,decoder_start_token_id,0
9,pad_token_id,0


In [16]:
from contextlib import nullcontext

rouge_validation_indices = [
    7, 8, 12, 17, 36,
    45, 47, 49, 54, 61,
    63, 68, 71, 73, 75,
    83, 84, 86, 89, 94
]

EVAL_BATCH_SIZE = 4

diagnostic_dataset = tokenized_dataset["validation"].select(
    rouge_validation_indices
)

evaluation_batches = []

for i in range(0, len(diagnostic_dataset), EVAL_BATCH_SIZE):

    batch_samples = [
        {
            "input_ids": diagnostic_dataset[j]["input_ids"],
            "attention_mask": diagnostic_dataset[j]["attention_mask"]
        }
        for j in range(
            i,
            min(i + EVAL_BATCH_SIZE, len(diagnostic_dataset))
        )
    ]

    batch = tokenizer.pad(
        batch_samples,
        padding=True,
        return_tensors="pt"
    )

    evaluation_batches.append(batch)

print("Validation Samples:", len(diagnostic_dataset))
print("Evaluation Batches:", len(evaluation_batches))

Validation Samples: 20
Evaluation Batches: 5


In [18]:
def generate_diagnostic_outputs(model, disable_lora=False):

    model.eval()

    generated_rows = []

    adapter_context = (
        model.disable_adapter()
        if disable_lora
        else nullcontext()
    )

    with adapter_context, torch.inference_mode():

        for batch in evaluation_batches:

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
            }

            precision_context = (
                torch.autocast("cuda", dtype=torch.bfloat16)
                if device == "cuda"
                else nullcontext()
            )

            with precision_context:

                generated = model.generate(
                    **inputs,
                    max_length= 768,
                    num_beams=1,
                    do_sample= False , 
                    repetition_penalty=1.0,
                    no_repeat_ngram_size= 0
                )

            generated_rows.extend(
                generated.cpu().tolist()
            )

    return generated_rows

In [19]:
generate_diagnostic_outputs(model, disable_lora=False)

[[0,
  37,
  3,
  7316,
  13167,
  33,
  3,
  9,
  3,
  7316,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  3,
  9,
  3,
  23098,
  13,
  

In [20]:
zero_shot_generated_ids = generate_diagnostic_outputs(
    model,
    disable_lora=True
)

lora_generated_ids = generate_diagnostic_outputs(
    model,
    disable_lora=False
)

print("Zero-Shot Outputs:", len(zero_shot_generated_ids))
print("LoRA Outputs:", len(lora_generated_ids))

Zero-Shot Outputs: 20
LoRA Outputs: 20


In [22]:
special_token_ids = set(tokenizer.all_special_ids)

eos_token_id = tokenizer.eos_token_id

forced_eos_token_id = model.generation_config.forced_eos_token_id

def analyze_generation(token_ids):

    eos_positions = [
        i
        for i, token_id in enumerate(token_ids)
        if i > 0 and token_id == eos_token_id
    ]

    eos_position = (
        eos_positions[0]
        if eos_positions
        else None
    )

    effective_ids = (
        token_ids[:eos_position + 1]
        if eos_position is not None
        else token_ids
    )

    content_ids = [
        token_id
        for token_id in effective_ids[1:]
        if token_id not in special_token_ids
    ]

    generated_tokens = len(content_ids)

    eos_before_limit = (
        eos_position is not None
        and eos_position < GENERATION_MAX_LENGTH - 1
    )

    reached_max_length = (
        len(effective_ids) >= GENERATION_MAX_LENGTH
    )

    decoded_text = tokenizer.decode(
        effective_ids,
        skip_special_tokens=True
    )

    empty_output = not decoded_text.strip()

    return {
        "generated_tokens": generated_tokens,
        "eos_generated": eos_position is not None,
        "eos_position": eos_position,
        "eos_before_limit": eos_before_limit,
        "reached_max_length": reached_max_length,
        "empty_output": empty_output,
        "content_ids": content_ids,
        "generated_text": decoded_text
    }



diagnostic_records = []

for model_name, generated_outputs in [
    ("zero_shot", zero_shot_generated_ids),
    ("lora", lora_generated_ids)
]:

    for sample_id, token_ids in zip(
        rouge_validation_indices,
        generated_outputs
    ):

        result = analyze_generation(token_ids)

        diagnostic_records.append({
            "sample_id": sample_id,
            "model": model_name,
            **result
        })

generation_diagnostics_df = pd.DataFrame(
    diagnostic_records
)

display(generation_diagnostics_df[[
    "sample_id",
    "model",
    "generated_tokens",
    "eos_generated",
    "eos_position",
    "eos_before_limit",
    "reached_max_length",
    "empty_output"
]])

print("Forced EOS Token:", forced_eos_token_id)
print("Total Records:", len(generation_diagnostics_df))

,sample_id,model,generated_tokens,eos_generated,eos_position,eos_before_limit,reached_max_length,empty_output
0,7,zero_shot,18,True,19.0,True,False,False
1,8,zero_shot,13,True,14.0,True,False,False
2,12,zero_shot,30,True,31.0,True,False,False
3,17,zero_shot,34,True,35.0,True,False,False
4,36,zero_shot,23,True,24.0,True,False,False
5,45,zero_shot,31,True,32.0,True,False,False
6,47,zero_shot,33,True,34.0,True,False,False
7,49,zero_shot,26,True,27.0,True,False,False
8,54,zero_shot,7,True,8.0,True,False,False
9,61,zero_shot,17,True,18.0,True,False,False


Forced EOS Token: None
Total Records: 40


In [23]:
generation_length_summary = (
    generation_diagnostics_df
    .groupby("model")
    .agg(
        mean_generated_tokens=("generated_tokens", "mean"),
        median_generated_tokens=("generated_tokens", "median"),
        maximum_generated_tokens=("generated_tokens", "max"),
        eos_before_limit_rate=("eos_before_limit", "mean"),
        reached_max_length_rate=("reached_max_length", "mean"),
        empty_output_rate=("empty_output", "mean")
    )
)

rate_columns = [
    "eos_before_limit_rate",
    "reached_max_length_rate",
    "empty_output_rate"
]

generation_length_summary[rate_columns] *= 100

display(generation_length_summary.T)

model,lora,zero_shot
mean_generated_tokens,752.4,61.55
median_generated_tokens,767.0,25.50
maximum_generated_tokens,767.0,767.00
eos_before_limit_rate,10.0,95.00
reached_max_length_rate,90.0,5.00
empty_output_rate,0.0,0.00


In [24]:
from collections import Counter

def create_ngrams(tokens, n):

    return [
        tuple(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    ]

In [25]:
def calculate_repetition_metrics(tokens):

    unigrams = create_ngrams(tokens, 1)
    bigrams = create_ngrams(tokens, 2)
    trigrams = create_ngrams(tokens, 3)

    distinct_1 = (
        len(set(unigrams)) / len(unigrams)
        if unigrams
        else 0.0
    )

    distinct_2 = (
        len(set(bigrams)) / len(bigrams)
        if bigrams
        else 0.0
    )

    repeated_trigram_ratio = (
        1 - len(set(trigrams)) / len(trigrams)
        if trigrams
        else 0.0
    )

    trigram_counts = Counter(trigrams)

    max_repeated_ngram_frequency = (
        max(trigram_counts.values())
        if trigram_counts
        else 0
    )

    return {
        "distinct_1": distinct_1,
        "distinct_2": distinct_2,
        "repeated_trigram_ratio": repeated_trigram_ratio,
        "max_repeated_ngram_frequency": max_repeated_ngram_frequency
    }

In [27]:
repetition_metrics_df = pd.DataFrame(
    generation_diagnostics_df["content_ids"]
    .apply(calculate_repetition_metrics)
    .tolist()
)

generation_diagnostics_df[
    repetition_metrics_df.columns
] = repetition_metrics_df

display(generation_diagnostics_df[[
    "sample_id",
    "model",
    "generated_tokens",
    "distinct_1",
    "distinct_2",
    "repeated_trigram_ratio",
    "max_repeated_ngram_frequency",
    "eos_generated",
    "reached_max_length"
]])

,sample_id,model,generated_tokens,distinct_1,distinct_2,repeated_trigram_ratio,max_repeated_ngram_frequency,eos_generated,reached_max_length
0,7,zero_shot,18,0.777778,1.000000,0.000000,1,True,False
1,8,zero_shot,13,0.846154,1.000000,0.000000,1,True,False
2,12,zero_shot,30,0.833333,0.965517,0.000000,1,True,False
3,17,zero_shot,34,0.911765,1.000000,0.000000,1,True,False
4,36,zero_shot,23,0.739130,1.000000,0.000000,1,True,False
5,45,zero_shot,31,0.741935,0.900000,0.034483,2,True,False
6,47,zero_shot,33,0.939394,1.000000,0.000000,1,True,False
7,49,zero_shot,26,1.000000,1.000000,0.000000,1,True,False
8,54,zero_shot,7,1.000000,1.000000,0.000000,1,True,False
9,61,zero_shot,17,1.000000,1.000000,0.000000,1,True,False


In [28]:
repetition_summary = (
    generation_diagnostics_df
    .groupby("model")
    .agg(
        mean_distinct_1=("distinct_1", "mean"),
        mean_distinct_2=("distinct_2", "mean"),
        mean_repeated_trigram_ratio=(
            "repeated_trigram_ratio", "mean"
        ),
        mean_max_ngram_frequency=(
            "max_repeated_ngram_frequency", "mean"
        ),
        maximum_ngram_frequency=(
            "max_repeated_ngram_frequency", "max"
        )
    )
)

display(repetition_summary.T)

model,lora,zero_shot
mean_distinct_1,0.032759,0.827806
mean_distinct_2,0.040897,0.931898
mean_repeated_trigram_ratio,0.956754,0.057902
mean_max_ngram_frequency,54.850000,3.900000
maximum_ngram_frequency,152.000000,55.000000


In [31]:
print("Forced EOS Token:", forced_eos_token_id)

generation_diagnostics_df["eos_status"] = np.select(
    [
        generation_diagnostics_df["eos_before_limit"],
        generation_diagnostics_df["eos_generated"]
        & generation_diagnostics_df["reached_max_length"],
        ~generation_diagnostics_df["eos_generated"]
        & generation_diagnostics_df["reached_max_length"]
    ],
    [
        "eos_before_limit",
        "eos_at_limit",
        "max_length_without_eos"
    ],
    default="other"
)

generation_diagnostics_df["forced_eos_token_id"] = (
    forced_eos_token_id
)

display(pd.crosstab(
    generation_diagnostics_df["model"],
    generation_diagnostics_df["eos_status"]
))

Forced EOS Token: None


eos_status,eos_before_limit,max_length_without_eos
model,,
lora,2,18
zero_shot,19,1


In [32]:
generation_diagnostics_df["raw_token_ids"] = (
    zero_shot_generated_ids + lora_generated_ids
)

generation_diagnostics_df.to_csv(
    reports_dir / "generation_diagnostics.csv",
    index=False
)

print("Total Records:", len(generation_diagnostics_df))
print("Report saved successfully.")

Total Records: 40
Report saved successfully.


In [33]:
decoding_configs = {
    "A": {
        "max_length": 768,
        "num_beams": 1,
        "repetition_penalty": 1.0,
        "no_repeat_ngram_size": 0,
        "do_sample": False
    },
    "B": {
        "max_length": 256,
        "num_beams": 1 ,
        "repetition_penalty": 1.0,
        "no_repeat_ngram_size": 0,
        "do_sample": False
    },
    "C": {
        "max_length": 256,
        "num_beams": 1,
        "repetition_penalty": 1.0,
        "no_repeat_ngram_size": 3,
        "do_sample": False
    }
}

decoding_configs_df = pd.DataFrame(
    decoding_configs
).T

display(decoding_configs_df)

,max_length,num_beams,repetition_penalty,no_repeat_ngram_size,do_sample
A,768,1,1.0,0,False
B,256,1,1.0,0,False
C,256,1,1.0,3,False


In [34]:
def generate_with_config(model, config, disable_lora=False):

    model.eval()

    generated_rows = []

    adapter_context = (
        model.disable_adapter()
        if disable_lora
        else nullcontext()
    )

    with adapter_context, torch.inference_mode():

        for batch in evaluation_batches:

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
            }

            precision_context = (
                torch.autocast("cuda", dtype=torch.bfloat16)
                if device == "cuda"
                else nullcontext()
            )

            with precision_context:

                generated = model.generate(
                    **inputs,
                    **config,
                    max_new_tokens=None
                )

            generated_rows.extend(
                generated.cpu().tolist()
            )

    return generated_rows

In [35]:
def analyze_generation(
    token_ids,
    max_length=GENERATION_MAX_LENGTH
):

    eos_positions = [
        i
        for i, token_id in enumerate(token_ids)
        if i > 0 and token_id == eos_token_id
    ]

    eos_position = (
        eos_positions[0]
        if eos_positions
        else None
    )

    effective_ids = (
        token_ids[:eos_position + 1]
        if eos_position is not None
        else token_ids
    )

    content_ids = [
        token_id
        for token_id in effective_ids[1:]
        if token_id not in special_token_ids
    ]

    generated_tokens = len(content_ids)

    eos_before_limit = (
        eos_position is not None
        and eos_position < max_length - 1
    )

    reached_max_length = (
        len(effective_ids) >= max_length
    )

    decoded_text = tokenizer.decode(
        effective_ids,
        skip_special_tokens=True
    )

    empty_output = not decoded_text.strip()

    return {
        "generated_tokens": generated_tokens,
        "eos_generated": eos_position is not None,
        "eos_position": eos_position,
        "eos_before_limit": eos_before_limit,
        "reached_max_length": reached_max_length,
        "empty_output": empty_output,
        "content_ids": content_ids,
        "generated_text": decoded_text
    }

In [36]:
import time

decoding_outputs = {}

experiment_start_time = time.perf_counter()

for model_name, disable_lora in [
    ("zero_shot", True),
    ("lora", False)
]:

    for config_name, config in decoding_configs.items():

        print(
            f"Running: {model_name} | Config: {config_name}"
        )

        generated_outputs = generate_with_config(
            model=model,
            config=config,
            disable_lora=disable_lora
        )

        decoding_outputs[
            (model_name, config_name)
        ] = generated_outputs

        print("Generated Outputs:", len(generated_outputs))

total_experiment_time = (
    time.perf_counter() - experiment_start_time
)

total_generations = sum(
    len(outputs)
    for outputs in decoding_outputs.values()
)

print("\nTotal Generations:", total_generations)
print("Total Experiment Time:", total_experiment_time)

assert total_generations == 120

Running: zero_shot | Config: A
Generated Outputs: 20
Running: zero_shot | Config: B
Generated Outputs: 20
Running: zero_shot | Config: C
Generated Outputs: 20
Running: lora | Config: A
Generated Outputs: 20
Running: lora | Config: B
Generated Outputs: 20
Running: lora | Config: C
Generated Outputs: 20

Total Generations: 120
Total Experiment Time: 51.2852272250002


In [37]:
import evaluate

rouge_metric = evaluate.load("rouge")

reference_summaries = tokenizer.batch_decode(
    [
        [
            token_id if token_id != -100
            else tokenizer.pad_token_id
            for token_id in sample["labels"]
        ]
        for sample in diagnostic_dataset
    ],
    skip_special_tokens=True
)

print("Reference Summaries:", len(reference_summaries))

Reference Summaries: 20


In [38]:
decoding_records = []

for (model_name, config_name), outputs in decoding_outputs.items():

    config = decoding_configs[config_name]

    for sample_id, token_ids, reference in zip(
        rouge_validation_indices,
        outputs,
        reference_summaries
    ):

        generation_result = analyze_generation(
            token_ids,
            max_length=config["max_length"]
        )

        content_ids = generation_result["content_ids"]

        repetition_result = calculate_repetition_metrics(
            content_ids
        )

        trigrams = create_ngrams(content_ids, 3)

        repeated_trigram_count = (
            len(trigrams) - len(set(trigrams))
        )

        decoding_records.append({
            "sample_id": sample_id,
            "model": model_name,
            "config": config_name,
            **config,
            **generation_result,
            **repetition_result,
            "repeated_trigram_count": repeated_trigram_count,
            "reference_summary": reference,
            "raw_token_ids": token_ids
        })

decoding_details_df = pd.DataFrame(decoding_records)

assert len(decoding_details_df) == 120

display(decoding_details_df[[
    "sample_id",
    "model",
    "config",
    "generated_tokens",
    "distinct_1",
    "distinct_2",
    "repeated_trigram_ratio",
    "repeated_trigram_count",
    "eos_generated",
    "reached_max_length"
]])

,sample_id,model,config,generated_tokens,distinct_1,distinct_2,repeated_trigram_ratio,repeated_trigram_count,eos_generated,reached_max_length
0,7,zero_shot,A,18,0.777778,1.000000,0.0,0,True,False
1,8,zero_shot,A,13,0.846154,1.000000,0.0,0,True,False
2,12,zero_shot,A,30,0.833333,0.965517,0.0,0,True,False
3,17,zero_shot,A,34,0.911765,1.000000,0.0,0,True,False
4,36,zero_shot,A,23,0.739130,1.000000,0.0,0,True,False
...,...,...,...,...,...,...,...,...,...,...
115,83,lora,C,255,0.368627,0.759843,0.0,0,False,True
116,84,lora,C,255,0.423529,0.740157,0.0,0,False,True
117,86,lora,C,143,0.426573,0.732394,0.0,0,True,False
118,89,lora,C,185,0.475676,0.771739,0.0,0,True,False


In [39]:
decoding_summary_records = []

for (model_name, config_name), group in decoding_details_df.groupby(
    ["model", "config"]
):

    rouge_scores = rouge_metric.compute(
        predictions=group["generated_text"].tolist(),
        references=group["reference_summary"].tolist(),
        use_stemmer=True
    )

    decoding_summary_records.append({
        "model": model_name,
        "config": config_name,
        "sample_count": len(group),
        "rouge1": rouge_scores["rouge1"] * 100,
        "rouge2": rouge_scores["rouge2"] * 100,
        "rougeL": rouge_scores["rougeL"] * 100,
        "avg_tokens": group["generated_tokens"].mean(),
        "mean_distinct_1": group["distinct_1"].mean(),
        "mean_distinct_2": group["distinct_2"].mean(),
        "repeated_trigram_ratio": group[
            "repeated_trigram_ratio"
        ].mean(),
        "total_repeated_trigram_count": group[
            "repeated_trigram_count"
        ].sum(),
        "mean_repeated_trigram_count": group[
            "repeated_trigram_count"
        ].mean(),
        "max_trigram_frequency": group[
            "max_repeated_ngram_frequency"
        ].max(),
        "max_length_rate": group[
            "reached_max_length"
        ].mean() * 100,
        "eos_before_limit_rate": group[
            "eos_before_limit"
        ].mean() * 100,
        "empty_output_rate": group[
            "empty_output"
        ].mean() * 100
    })

decoding_experiment_df = pd.DataFrame(
    decoding_summary_records
)

display(decoding_experiment_df)

,model,config,sample_count,rouge1,rouge2,rougeL,avg_tokens,mean_distinct_1,mean_distinct_2,repeated_trigram_ratio,total_repeated_trigram_count,mean_repeated_trigram_count,max_trigram_frequency,max_length_rate,eos_before_limit_rate,empty_output_rate
0,lora,A,20,14.197240,1.478052,13.274526,752.40,0.032759,0.040897,0.956754,14366,718.30,152,90.0,10.0,0.0
1,lora,B,20,19.979337,2.353645,14.816698,254.65,0.095527,0.119795,0.872808,4411,220.55,50,100.0,0.0,0.0
2,lora,C,20,26.044892,5.502064,14.671474,201.25,0.437410,0.760291,0.000000,0,0.00,1,25.0,75.0,0.0
3,zero_shot,A,20,5.163355,1.379274,4.262540,61.55,0.827806,0.931898,0.057902,746,37.30,55,5.0,95.0,0.0
4,zero_shot,B,20,5.415893,1.401198,4.325671,35.95,0.829900,0.934792,0.054728,234,11.70,18,5.0,95.0,0.0
5,zero_shot,C,20,5.140493,1.484483,4.085911,27.45,0.858743,0.981943,0.000000,0,0.00,1,0.0,100.0,0.0


In [41]:
display(decoding_experiment_df[[
    "model",
    "config",
    "rougeL",
    "avg_tokens",
    "repeated_trigram_ratio",
    "max_length_rate",
    "eos_before_limit_rate"
]].round(6))

decoding_experiment_df.to_csv(
    reports_dir / "decoding_experiment.csv",
    index=False
)

print("Experiment Records:", len(decoding_experiment_df))
print("Report saved successfully.")

,model,config,rougeL,avg_tokens,repeated_trigram_ratio,max_length_rate,eos_before_limit_rate
0,lora,A,13.274526,752.40,0.956754,90.0,10.0
1,lora,B,14.816698,254.65,0.872808,100.0,0.0
2,lora,C,14.671474,201.25,0.000000,25.0,75.0
3,zero_shot,A,4.262540,61.55,0.057902,5.0,95.0
4,zero_shot,B,4.325671,35.95,0.054728,5.0,95.0
5,zero_shot,C,4.085911,27.45,0.000000,0.0,100.0


Experiment Records: 6
Report saved successfully.


In [42]:
experiment_index = decoding_experiment_df.set_index(
    ["model", "config"]
)

hypothesis_1_records = []

for model_name in ["zero_shot", "lora"]:

    config_a = experiment_index.loc[(model_name, "A")]
    config_b = experiment_index.loc[(model_name, "B")]

    repeat_a = config_a["total_repeated_trigram_count"]
    repeat_b = config_b["total_repeated_trigram_count"]

    hypothesis_1_records.append({
        "model": model_name,
        "config_a_repeated_trigrams": repeat_a,
        "config_b_repeated_trigrams": repeat_b,
        "absolute_reduction": repeat_a - repeat_b,
        "reduction_percentage": (
            (repeat_a - repeat_b) / repeat_a * 100
            if repeat_a > 0
            else 0.0
        ),
        "hypothesis_supported": repeat_b < repeat_a
    })

hypothesis_1_df = pd.DataFrame(hypothesis_1_records)

display(hypothesis_1_df)

,model,config_a_repeated_trigrams,config_b_repeated_trigrams,absolute_reduction,reduction_percentage,hypothesis_supported
0,zero_shot,746.0,234.0,512.0,68.632708,True
1,lora,14366.0,4411.0,9955.0,69.295559,True


In [43]:
hypothesis_2_records = []

for model_name in ["zero_shot", "lora"]:

    config_b = experiment_index.loc[(model_name, "B")]
    config_c = experiment_index.loc[(model_name, "C")]

    ratio_b = config_b["repeated_trigram_ratio"]
    ratio_c = config_c["repeated_trigram_ratio"]

    hypothesis_2_records.append({
        "model": model_name,
        "config_b_repetition_ratio": ratio_b,
        "config_c_repetition_ratio": ratio_c,
        "absolute_reduction": ratio_b - ratio_c,
        "reduction_percentage": (
            (ratio_b - ratio_c) / ratio_b * 100
            if ratio_b > 0
            else 0.0
        ),
        "hypothesis_supported": ratio_c < ratio_b
    })

hypothesis_2_df = pd.DataFrame(hypothesis_2_records)

display(hypothesis_2_df)

,model,config_b_repetition_ratio,config_c_repetition_ratio,absolute_reduction,reduction_percentage,hypothesis_supported
0,zero_shot,0.054728,0.0,0.054728,100.0,True
1,lora,0.872808,0.0,0.872808,100.0,True


In [44]:
hypothesis_3_records = []

for model_name in ["zero_shot", "lora"]:

    config_b = experiment_index.loc[(model_name, "B")]
    config_c = experiment_index.loc[(model_name, "C")]

    repetition_decreased = (
        config_c["repeated_trigram_ratio"]
        < config_b["repeated_trigram_ratio"]
    )

    hypothesis_3_records.append({
        "model": model_name,
        "repetition_decreased": repetition_decreased,
        "rouge1_change": (
            config_c["rouge1"] - config_b["rouge1"]
        ),
        "rouge2_change": (
            config_c["rouge2"] - config_b["rouge2"]
        ),
        "rougeL_change": (
            config_c["rougeL"] - config_b["rougeL"]
        ),
        "hypothesis_supported": (
            repetition_decreased
            and config_c["rougeL"] <= config_b["rougeL"]
        )
    })

hypothesis_3_df = pd.DataFrame(hypothesis_3_records)

display(hypothesis_3_df)

,model,repetition_decreased,rouge1_change,rouge2_change,rougeL_change,hypothesis_supported
0,zero_shot,True,-0.275400,0.083284,-0.239760,True
1,lora,True,6.065556,3.148419,-0.145224,True


In [51]:
factuality_indices = [7, 36, 61, 75, 94]

factuality_configs = ["A", "C"]

In [52]:
factuality_df = decoding_details_df[
    decoding_details_df["sample_id"].isin(factuality_indices)
    & decoding_details_df["config"].isin(factuality_configs)
].copy()

factuality_df["source_text"] = factuality_df["sample_id"].apply(
    lambda sample_id: tokenizer.decode(
        tokenized_dataset["validation"][int(sample_id)]["input_ids"],
        skip_special_tokens=True
    )
)

factuality_df = factuality_df.sort_values(
    ["sample_id", "model", "config"]
).reset_index(drop=True)

assert len(factuality_df) == 20

display(factuality_df[[
    "sample_id",
    "model",
    "config",
    "generated_tokens",
    "repeated_trigram_ratio",
    "eos_generated",
    "reached_max_length"
]])

,sample_id,model,config,generated_tokens,repeated_trigram_ratio,eos_generated,reached_max_length
0,7,lora,A,767,0.983007,False,True
1,7,lora,C,217,0.000000,True,False
2,7,zero_shot,A,18,0.000000,True,False
3,7,zero_shot,C,18,0.000000,True,False
4,36,lora,A,767,0.969935,False,True
5,36,lora,C,255,0.000000,False,True
6,36,zero_shot,A,23,0.000000,True,False
7,36,zero_shot,C,23,0.000000,True,False
8,61,lora,A,767,0.952941,False,True
9,61,lora,C,159,0.000000,True,False


In [53]:
for sample_id in factuality_indices:

    sample_group = factuality_df[
        factuality_df["sample_id"] == sample_id
    ]

    print("=" * 100)
    print("VALIDATION ID:", sample_id)

    print("\nSOURCE TEXT:\n")
    print(sample_group.iloc[0]["source_text"])

    print("\nREFERENCE SUMMARY:\n")
    print(sample_group.iloc[0]["reference_summary"])

    for _, row in sample_group.iterrows():

        print("\n" + "-" * 80)
        print("MODEL:", row["model"])
        print("CONFIG:", row["config"])
        print("GENERATED TOKENS:", row["generated_tokens"])
        print("REPETITION RATIO:", row["repeated_trigram_ratio"])
        print("EOS GENERATED:", row["eos_generated"])
        print("REACHED MAX LENGTH:", row["reached_max_length"])

        print("\nGENERATED SUMMARY:\n")
        print(row["generated_text"])

    print("=" * 100)

VALIDATION ID: 7

SOURCE TEXT:

Summarize the following biomedical research article in plain language for a general audience: Inheritance of gene expression states is fundamental for cells to ‘remember’ past events , such as environmental or developmental cues . The conserved Polycomb Repressive Complex 2 ( PRC2 ) maintains epigenetic repression of many genes in animals and plants and modifies chromatin at its targets . Histones modified by PRC2 can be inherited through cell division . However , it remains unclear whether this inheritance can direct long-term memory of individual gene expression states ( cis memory ) or instead if local chromatin states are dictated by the concentrations of diffusible factors ( trans memory ) . By monitoring the expression of two copies of the Arabidopsis Polycomb target gene FLOWERING LOCUS C ( FLC ) in the same plants , we show that one copy can be repressed while the other is active . Furthermore , this ‘mixed’ expression state is inherited through 

In [57]:
file_path = reports_dir / "factual_consistency_analysis.csv"

print("File Path:", file_path)
print("File Exists:", file_path.exists())

File Path: /home/pasali/Desktop/Data/AI-Engineer/Hafta-7/HealthBrief/reports/factual_consistency_analysis.csv
File Exists: True


In [58]:
factual_consistency_df = pd.read_csv(file_path)

display(factual_consistency_df[[
    "sample_id",
    "model",
    "config",
    "claim",
    "factuality",
    "evidence"
]])

print("Total Records:", len(factual_consistency_df))

,sample_id,model,config,claim,factuality,evidence
0,7,lora,A,Repeated molecule phrase; no stable scientific...,uninterpretable,NaN
1,7,lora,C,The FLC fusion proteins are a fusion of a sing...,contradicted,the Venus or mCherry sequence inserted as a tr...
2,7,lora,C,Phospholysis is inherited by cells.,not_verifiable,NaN
3,7,lora,C,Phosphatase is regulated by an unspecified enz...,not_verifiable,NaN
4,7,lora,C,Other phosphate/ATP/phosphopeptide clauses lac...,uninterpretable,NaN
5,7,zero_shot,A,The FLC fusion proteins are functional.,supported,demonstrating that the fusion proteins are fun...
6,7,zero_shot,C,The FLC fusion proteins are functional.,supported,demonstrating that the fusion proteins are fun...
7,36,lora,A,Microbial community changes can be used to est...,supported,allowing PMI to be estimated within approximat...
8,36,lora,C,Microbial community changes can be used to est...,supported,allowing PMI to be estimated within approximat...
9,36,lora,C,The study used a mouse model.,supported,"By using a mouse model , we were able to perfo..."


Total Records: 41


In [59]:
factuality_summary = pd.crosstab(
    [
        factual_consistency_df["model"],
        factual_consistency_df["config"]
    ],
    factual_consistency_df["factuality"]
)

factuality_summary["assessable_claims"] = (
    factuality_summary["supported"]
    + factuality_summary["contradicted"]
    + factuality_summary["not_verifiable"]
)

for column in [
    "supported",
    "contradicted",
    "not_verifiable"
]:

    factuality_summary[f"{column}_rate"] = (
        factuality_summary[column]
        / factuality_summary["assessable_claims"].replace(0, np.nan)
        * 100
    )

display(factuality_summary)

factuality        contradicted  not_verifiable  supported  uninterpretable  \
model     config                                                             
lora      A                  0               2          1                2   
          C                  1              12          6                5   
zero_shot A                  1               0          4                1   
          C                  1               0          4                1   

factuality        assessable_claims  supported_rate  contradicted_rate  \
model     config                                                         
lora      A                       3       33.333333           0.000000   
          C                      19       31.578947           5.263158   
zero_shot A                       5       80.000000          20.000000   
          C                       5       80.000000          20.000000   

factuality        not_verifiable_rate  
model     config                       
lora      A                 66.666667  
          C                 63.157895  
zero_shot A                  0.000000  
          C                  0.000000

In [60]:
summary_level_df = factual_consistency_df.drop_duplicates(
    subset=["sample_id", "model", "config"]
)

scientific_error_summary = (
    summary_level_df
    .groupby(["model", "config"])
    .agg(
        summary_count=("sample_id", "count"),
        unsupported_claims=("unsupported_scientific_claim", "sum"),
        contradictions=("contradiction", "sum"),
        missing_key_findings=("missing_key_finding", "sum"),
        terminology_errors=(
            "terminology_error",
            lambda values: (values == "yes").sum()
        ),
        uncertainty_errors=(
            "uncertainty_error",
            lambda values: (values == "yes").sum()
        ),
        severe_repetition=("severe_repetition", "sum"),
        incomplete_summaries=("incomplete_summary", "sum"),
        key_findings_covered=("key_findings_covered", "sum"),
        key_findings_total=("key_findings_total", "sum")
    )
)

scientific_error_summary["key_finding_coverage_rate"] = (
    scientific_error_summary["key_findings_covered"]
    / scientific_error_summary["key_findings_total"]
    * 100
)

display(scientific_error_summary.T)

model                           lora             zero_shot           
config                             A          C          A          C
summary_count               5.000000   5.000000   5.000000   5.000000
unsupported_claims          2.000000   5.000000   0.000000   0.000000
contradictions              0.000000   1.000000   1.000000   1.000000
missing_key_findings        5.000000   5.000000   5.000000   5.000000
terminology_errors          2.000000   5.000000   1.000000   1.000000
uncertainty_errors          0.000000   0.000000   0.000000   0.000000
severe_repetition           5.000000   0.000000   1.000000   0.000000
incomplete_summaries        5.000000   5.000000   5.000000   5.000000
key_findings_covered        1.000000   1.000000   1.000000   1.000000
key_findings_total         15.000000  15.000000  15.000000  15.000000
key_finding_coverage_rate   6.666667   6.666667   6.666667   6.666667

In [61]:
automatic_metrics_df = decoding_experiment_df[
    decoding_experiment_df["config"].isin(["A", "C"])
][[
    "model",
    "config",
    "sample_count",
    "rouge1",
    "rouge2",
    "rougeL",
    "mean_distinct_2",
    "repeated_trigram_ratio",
    "eos_before_limit_rate"
]].copy()

automatic_metrics_df = automatic_metrics_df.rename(
    columns={
        "sample_count": "automatic_sample_count",
        "mean_distinct_2": "distinct_2"
    }
)

display(automatic_metrics_df)

,model,config,automatic_sample_count,rouge1,rouge2,rougeL,distinct_2,repeated_trigram_ratio,eos_before_limit_rate
0,lora,A,20,14.197240,1.478052,13.274526,0.040897,0.956754,10.0
2,lora,C,20,26.044892,5.502064,14.671474,0.760291,0.000000,75.0
3,zero_shot,A,20,5.163355,1.379274,4.262540,0.931898,0.057902,95.0
5,zero_shot,C,20,5.140493,1.484483,4.085911,0.981943,0.000000,100.0


In [62]:
claim_metrics_df = factuality_summary.reset_index()[[
    "model",
    "config",
    "supported",
    "contradicted",
    "not_verifiable",
    "uninterpretable",
    "assessable_claims",
    "supported_rate",
    "contradicted_rate",
    "not_verifiable_rate"
]].copy()

claim_metrics_df = claim_metrics_df.rename(
    columns={
        "supported": "supported_claim_count",
        "contradicted": "contradiction_count",
        "supported_rate": "supported_claim_rate"
    }
)

display(claim_metrics_df)

factuality,model,config,supported_claim_count,contradiction_count,not_verifiable,uninterpretable,assessable_claims,supported_claim_rate,contradicted_rate,not_verifiable_rate
0,lora,A,1,0,2,2,3,33.333333,0.000000,66.666667
1,lora,C,6,1,12,5,19,31.578947,5.263158,63.157895
2,zero_shot,A,4,1,0,1,5,80.000000,20.000000,0.000000
3,zero_shot,C,4,1,0,1,5,80.000000,20.000000,0.000000


In [63]:
scientific_metrics_df = scientific_error_summary.reset_index()[[
    "model",
    "config",
    "summary_count",
    "terminology_errors",
    "severe_repetition",
    "incomplete_summaries",
    "key_findings_covered",
    "key_findings_total",
    "key_finding_coverage_rate"
]].copy()

scientific_metrics_df = scientific_metrics_df.rename(
    columns={
        "summary_count": "manual_sample_count"
    }
)

display(scientific_metrics_df)

,model,config,manual_sample_count,terminology_errors,severe_repetition,incomplete_summaries,key_findings_covered,key_findings_total,key_finding_coverage_rate
0,lora,A,5,2,5,5,1,15,6.666667
1,lora,C,5,5,0,5,1,15,6.666667
2,zero_shot,A,5,1,1,5,1,15,6.666667
3,zero_shot,C,5,1,0,5,1,15,6.666667


In [64]:
generation_quality_comparison = (
    automatic_metrics_df
    .merge(
        claim_metrics_df,
        on=["model", "config"],
        how="inner",
        validate="one_to_one"
    )
    .merge(
        scientific_metrics_df,
        on=["model", "config"],
        how="inner",
        validate="one_to_one"
    )
)

generation_quality_comparison = (
    generation_quality_comparison
    .sort_values(["model", "config"])
    .reset_index(drop=True)
)

assert len(generation_quality_comparison) == 4

display(generation_quality_comparison)

,model,config,automatic_sample_count,rouge1,rouge2,rougeL,distinct_2,repeated_trigram_ratio,eos_before_limit_rate,supported_claim_count,...,supported_claim_rate,contradicted_rate,not_verifiable_rate,manual_sample_count,terminology_errors,severe_repetition,incomplete_summaries,key_findings_covered,key_findings_total,key_finding_coverage_rate
0,lora,A,20,14.197240,1.478052,13.274526,0.040897,0.956754,10.0,1,...,33.333333,0.000000,66.666667,5,2,5,5,1,15,6.666667
1,lora,C,20,26.044892,5.502064,14.671474,0.760291,0.000000,75.0,6,...,31.578947,5.263158,63.157895,5,5,0,5,1,15,6.666667
2,zero_shot,A,20,5.163355,1.379274,4.262540,0.931898,0.057902,95.0,4,...,80.000000,20.000000,0.000000,5,1,1,5,1,15,6.666667
3,zero_shot,C,20,5.140493,1.484483,4.085911,0.981943,0.000000,100.0,4,...,80.000000,20.000000,0.000000,5,1,0,5,1,15,6.666667


In [65]:
comparison_columns = [
    "model",
    "config",
    "automatic_sample_count",
    "manual_sample_count",
    "rouge1",
    "rouge2",
    "rougeL",
    "distinct_2",
    "repeated_trigram_ratio",
    "eos_before_limit_rate",
    "supported_claim_rate",
    "contradiction_count",
    "key_finding_coverage_rate"
]

display(
    generation_quality_comparison[
        comparison_columns
    ].round(6)
)

,model,config,automatic_sample_count,manual_sample_count,rouge1,rouge2,rougeL,distinct_2,repeated_trigram_ratio,eos_before_limit_rate,supported_claim_rate,contradiction_count,key_finding_coverage_rate
0,lora,A,20,5,14.197240,1.478052,13.274526,0.040897,0.956754,10.0,33.333333,0,6.666667
1,lora,C,20,5,26.044892,5.502064,14.671474,0.760291,0.000000,75.0,31.578947,1,6.666667
2,zero_shot,A,20,5,5.163355,1.379274,4.262540,0.931898,0.057902,95.0,80.000000,1,6.666667
3,zero_shot,C,20,5,5.140493,1.484483,4.085911,0.981943,0.000000,100.0,80.000000,1,6.666667


In [66]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

from copy import deepcopy

repro_generation_config = deepcopy(
    model.generation_config
)

repro_generation_config.max_length = 768
repro_generation_config.max_new_tokens = None
repro_generation_config.num_beams = 1
repro_generation_config.do_sample = False
repro_generation_config.repetition_penalty = 1.0
repro_generation_config.no_repeat_ngram_size = 0

repro_training_args = Seq2SeqTrainingArguments(
    output_dir=str(models_dir / "healthbrief_lora"),
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    generation_max_length=768,
    generation_num_beams=1,
    generation_config=repro_generation_config,
    bf16=device == "cuda",
    fp16=False,
    eval_accumulation_steps=1,
    prediction_loss_only=False,
    seed=42,
    data_seed=42,
    report_to="none"
)

repro_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

In [67]:
def compute_rouge_metrics(eval_pred):

    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.where(
        predictions == -100,
        tokenizer.pad_token_id,
        predictions
    )

    labels = np.where(
        labels == -100,
        tokenizer.pad_token_id,
        labels
    )

    decoded_predictions = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    scores = rouge_metric.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        "rouge1": scores["rouge1"] * 100,
        "rouge2": scores["rouge2"] * 100,
        "rougeL": scores["rougeL"] * 100,
        "invalid_generation_count": sum(
            not text.strip()
            for text in decoded_predictions
        )
    }

In [68]:
class ReproducibilityTrainer(Seq2SeqTrainer):

    def __init__(self, *args, **kwargs):

        super().__init__(*args, **kwargs)

        self.captured_inputs = []
        self.captured_attention_masks = []
        self.captured_training_modes = []

    def prediction_step(
        self,
        model,
        inputs,
        prediction_loss_only,
        ignore_keys=None,
        **gen_kwargs
    ):

        self.captured_inputs.extend(
            inputs["input_ids"].detach().cpu().tolist()
        )

        self.captured_attention_masks.extend(
            inputs["attention_mask"].detach().cpu().tolist()
        )

        self.captured_training_modes.append(
            model.training
        )

        return super().prediction_step(
            model,
            inputs,
            prediction_loss_only,
            ignore_keys=ignore_keys,
            **gen_kwargs
        )

In [69]:
set_seed(SEED)

model.eval()

repro_trainer = ReproducibilityTrainer(
    model=model,
    args=repro_training_args,
    data_collator=repro_data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_rouge_metrics
)

trainer_prediction_results = repro_trainer.predict(
    diagnostic_dataset
)

print("Trainer Metrics:")
print(trainer_prediction_results.metrics)

print(
    "Prediction Shape:",
    trainer_prediction_results.predictions.shape
)

print(
    "Reference Shape:",
    trainer_prediction_results.label_ids.shape
)

Trainer Metrics:
{'test_loss': 2.7636990547180176, 'test_model_preparation_time': 0.0057, 'test_rouge1': 14.778890724073499, 'test_rouge2': 1.6853294303664654, 'test_rougeL': 13.751729578646366, 'test_invalid_generation_count': 0, 'test_runtime': 22.2962, 'test_samples_per_second': 0.897, 'test_steps_per_second': 0.224}
Prediction Shape: (20, 768)
Reference Shape: (20, 768)


In [70]:
manual_input_ids = [
    row
    for batch in evaluation_batches
    for row in batch["input_ids"].cpu().tolist()
]

manual_attention_masks = [
    row
    for batch in evaluation_batches
    for row in batch["attention_mask"].cpu().tolist()
]

input_ids_match = (
    manual_input_ids == repro_trainer.captured_inputs
)

attention_masks_match = (
    manual_attention_masks
    == repro_trainer.captured_attention_masks
)

trainer_eval_mode = not any(
    repro_trainer.captured_training_modes
)

print("Input IDs Match:", input_ids_match)
print("Attention Masks Match:", attention_masks_match)
print("Trainer Evaluation Mode:", trainer_eval_mode)
print("Trainer BF16:", repro_trainer.args.bf16)
print("Captured Samples:", len(repro_trainer.captured_inputs))

Input IDs Match: True
Attention Masks Match: True
Trainer Evaluation Mode: True
Trainer BF16: True
Captured Samples: 20


In [71]:
from itertools import zip_longest

def normalize_generated_ids(token_ids):

    normalized_ids = [
        tokenizer.pad_token_id if token_id == -100
        else int(token_id)
        for token_id in token_ids
    ]

    for position, token_id in enumerate(normalized_ids):

        if position > 0 and token_id == tokenizer.eos_token_id:

            return normalized_ids[:position + 1]

    return normalized_ids

In [72]:
trainer_generated_ids = (
    trainer_prediction_results.predictions
)

manual_generated_ids = decoding_outputs[("lora", "A")]

assert len(trainer_generated_ids) == len(manual_generated_ids) == 20

token_comparison_records = []

for sample_id, trainer_ids, manual_ids in zip(
    rouge_validation_indices,
    trainer_generated_ids,
    manual_generated_ids
):

    trainer_ids = normalize_generated_ids(trainer_ids)
    manual_ids = normalize_generated_ids(manual_ids)

    tokens_match = trainer_ids == manual_ids

    first_difference = next(
        (
            position
            for position, (trainer_token, manual_token)
            in enumerate(
                zip_longest(trainer_ids, manual_ids)
            )
            if trainer_token != manual_token
        ),
        None
    )

    token_comparison_records.append({
        "sample_id": sample_id,
        "trainer_length": len(trainer_ids),
        "manual_length": len(manual_ids),
        "tokens_match": tokens_match,
        "first_difference_position": first_difference,
        "trainer_different_token": (
            trainer_ids[first_difference]
            if first_difference is not None
            and first_difference < len(trainer_ids)
            else None
        ),
        "manual_different_token": (
            manual_ids[first_difference]
            if first_difference is not None
            and first_difference < len(manual_ids)
            else None
        )
    })

token_comparison_df = pd.DataFrame(
    token_comparison_records
)

display(token_comparison_df)

print(
    "Matching Outputs:",
    token_comparison_df["tokens_match"].sum()
)

print(
    "Different Outputs:",
    (~token_comparison_df["tokens_match"]).sum()
)

,sample_id,trainer_length,manual_length,tokens_match,first_difference_position,trainer_different_token,manual_different_token
0,7,768,768,True,NaN,NaN,NaN
1,8,768,768,True,NaN,NaN,NaN
2,12,747,747,True,NaN,NaN,NaN
3,17,768,768,True,NaN,NaN,NaN
4,36,768,768,True,NaN,NaN,NaN
5,45,768,768,False,2.0,3.0,454.0
6,47,768,768,True,NaN,NaN,NaN
7,49,768,768,False,5.0,8.0,3.0
8,54,768,768,True,NaN,NaN,NaN
9,61,768,768,True,NaN,NaN,NaN


Matching Outputs: 17
Different Outputs: 3


In [73]:
different_outputs = token_comparison_df[
    ~token_comparison_df["tokens_match"]
]

if not different_outputs.empty:

    first_difference = different_outputs.iloc[0]

    print("Sample ID:", first_difference["sample_id"])

    print(
        "First Difference Position:",
        first_difference["first_difference_position"]
    )

    for column in [
        "trainer_different_token",
        "manual_different_token"
    ]:

        token_id = first_difference[column]

        if pd.notna(token_id):

            token_id = int(token_id)

            print(column, "ID:", token_id)
            print(
                column,
                "Token:",
                tokenizer.convert_ids_to_tokens(token_id)
            )

        else:
            print(column, "Sequence ended")

else:

    print("All generated token sequences match.")

Sample ID: 45
First Difference Position: 2.0
trainer_different_token ID: 3
trainer_different_token Token: ▁
manual_different_token ID: 454
manual_different_token Token: ▁H


In [74]:
trainer_labels = np.where(
    trainer_prediction_results.label_ids == -100,
    tokenizer.pad_token_id,
    trainer_prediction_results.label_ids
)

trainer_references = tokenizer.batch_decode(
    trainer_labels,
    skip_special_tokens=True
)

reference_comparison_df = pd.DataFrame({
    "sample_id": rouge_validation_indices,
    "trainer_reference": trainer_references,
    "manual_reference": reference_summaries
})

reference_comparison_df["reference_match"] = (
    reference_comparison_df["trainer_reference"]
    == reference_comparison_df["manual_reference"]
)

display(reference_comparison_df[[
    "sample_id",
    "reference_match"
]])

reference_decoding_match = (
    reference_comparison_df["reference_match"].all()
)

print("Reference Decoding Match:", reference_decoding_match)

,sample_id,reference_match
0,7,True
1,8,True
2,12,True
3,17,True
4,36,True
5,45,True
6,47,True
7,49,True
8,54,True
9,61,True


Reference Decoding Match: True


In [75]:
generation_comparison_records = []

trainer_config = repro_trainer.model.generation_config

for parameter, manual_value in decoding_configs["A"].items():

    trainer_value = getattr(
        trainer_config,
        parameter,
        None
    )

    generation_comparison_records.append({
        "parameter": parameter,
        "trainer_value": trainer_value,
        "manual_value": manual_value,
        "match": trainer_value == manual_value
    })

generation_config_comparison = pd.DataFrame(
    generation_comparison_records
)

display(generation_config_comparison)

generation_config_match = (
    generation_config_comparison["match"].all()
)

print("Generation Config Match:", generation_config_match)

print("Trainer Forced EOS:", trainer_config.forced_eos_token_id)
print("Trainer Max New Tokens:", trainer_config.max_new_tokens)
print("Trainer Use Cache:", trainer_config.use_cache)

,parameter,trainer_value,manual_value,match
0,max_length,768,768,True
1,num_beams,1,1,True
2,repetition_penalty,1.0,1.0,True
3,no_repeat_ngram_size,0,0,True
4,do_sample,False,False,True


Generation Config Match: True
Trainer Forced EOS: None
Trainer Max New Tokens: None
Trainer Use Cache: True


In [76]:
with torch.autocast("cuda", dtype=torch.bfloat16):
    generated = model.generate(**inputs)

In [77]:
precision_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    revision=getattr(base_config, "_commit_hash", None)
)

precision_model = PeftModel.from_pretrained(
    precision_base_model,
    ADAPTER_PATH,
    is_trainable=False
)

precision_model.to(device)
precision_model.eval()

print("Model Dtype:", next(precision_model.parameters()).dtype)
print("Training Mode:", precision_model.training)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 13829.14it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model Dtype: torch.float32
Training Mode: False


In [78]:
def generate_with_precision(eval_model, use_bf16):

    eval_model.eval()

    generated_rows = []

    with torch.inference_mode():

        for batch in evaluation_batches:

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
            }

            with torch.autocast(
                "cuda",
                dtype=torch.bfloat16,
                enabled=use_bf16
            ):

                generated = eval_model.generate(
                    **inputs,
                    **decoding_configs["A"],
                    max_new_tokens=None
                )

            generated_rows.extend(
                generated.cpu().tolist()
            )

    return generated_rows

In [79]:
set_seed(SEED)

fp32_generated_ids = generate_with_precision(
    precision_model,
    use_bf16=False
)

set_seed(SEED)

bf16_generated_ids = generate_with_precision(
    precision_model,
    use_bf16=True
)

print("FP32 Outputs:", len(fp32_generated_ids))
print("BF16 Outputs:", len(bf16_generated_ids))

FP32 Outputs: 20
BF16 Outputs: 20


In [81]:
precision_comparison_records = []

for sample_id, trainer_ids, manual_ids, fp32_ids, bf16_ids in zip(
    rouge_validation_indices,
    trainer_generated_ids,
    manual_generated_ids,
    fp32_generated_ids,
    bf16_generated_ids
):

    trainer_ids = normalize_generated_ids(trainer_ids)
    manual_ids = normalize_generated_ids(manual_ids)
    fp32_ids = normalize_generated_ids(fp32_ids)
    bf16_ids = normalize_generated_ids(bf16_ids)

    precision_comparison_records.append({
        "sample_id": sample_id,
        "trainer_vs_manual": trainer_ids == manual_ids,
        "trainer_vs_fp32": trainer_ids == fp32_ids,
        "trainer_vs_bf16": trainer_ids == bf16_ids,
        "manual_vs_fp32": manual_ids == fp32_ids,
        "manual_vs_bf16": manual_ids == bf16_ids,
        "fp32_vs_bf16": fp32_ids == bf16_ids
    })

precision_comparison_df = pd.DataFrame(
    precision_comparison_records
)

display(precision_comparison_df)

precision_match_summary = (
    precision_comparison_df
    .drop(columns="sample_id")
    .sum()
    .to_frame("matching_samples")
)

precision_match_summary["match_rate"] = (
    precision_match_summary["matching_samples"]
    / len(precision_comparison_df)
    * 100
)

display(precision_match_summary)

,sample_id,trainer_vs_manual,trainer_vs_fp32,trainer_vs_bf16,manual_vs_fp32,manual_vs_bf16,fp32_vs_bf16
0,7,True,True,True,True,True,True
1,8,True,True,True,True,True,True
2,12,True,True,True,True,True,True
3,17,True,True,True,True,True,True
4,36,True,True,True,True,True,True
5,45,False,True,False,False,True,False
6,47,True,True,True,True,True,True
7,49,False,True,False,False,True,False
8,54,True,True,True,True,True,True
9,61,True,True,True,True,True,True


,matching_samples,match_rate
trainer_vs_manual,17,85.0
trainer_vs_fp32,20,100.0
trainer_vs_bf16,17,85.0
manual_vs_fp32,17,85.0
manual_vs_bf16,20,100.0
fp32_vs_bf16,17,85.0


In [82]:
precision_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    revision=getattr(base_config, "_commit_hash", None)
)

precision_model = PeftModel.from_pretrained(
    precision_base_model,
    ADAPTER_PATH,
    is_trainable=False
)

precision_model.generation_config = deepcopy(
    repro_generation_config
)

precision_model.to(device)
precision_model.eval()

print("Model Dtype:", next(precision_model.parameters()).dtype)
print("Training Mode:", precision_model.training)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 14475.73it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model Dtype: torch.float32
Training Mode: False


In [83]:
def generate_with_precision(eval_model, use_bf16):

    eval_model.eval()

    generated_rows = []

    with torch.inference_mode():

        for batch in evaluation_batches:

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
            }

            with torch.autocast(
                "cuda",
                dtype=torch.bfloat16,
                enabled=use_bf16
            ):

                generated = eval_model.generate(
                    **inputs,
                    **decoding_configs["A"],
                    max_new_tokens=None
                )

            generated_rows.extend(
                generated.cpu().tolist()
            )

    return generated_rows

In [84]:
set_seed(SEED)

fp32_generated_ids = generate_with_precision(
    precision_model,
    use_bf16=False
)

set_seed(SEED)

bf16_generated_ids = generate_with_precision(
    precision_model,
    use_bf16=True
)

print("FP32 Outputs:", len(fp32_generated_ids))
print("BF16 Outputs:", len(bf16_generated_ids))

FP32 Outputs: 20
BF16 Outputs: 20


In [85]:
precision_comparison_records = []

for sample_id, trainer_ids, manual_ids, fp32_ids, bf16_ids in zip(
    rouge_validation_indices,
    trainer_generated_ids,
    manual_generated_ids,
    fp32_generated_ids,
    bf16_generated_ids
):

    trainer_ids = normalize_generated_ids(trainer_ids)
    manual_ids = normalize_generated_ids(manual_ids)
    fp32_ids = normalize_generated_ids(fp32_ids)
    bf16_ids = normalize_generated_ids(bf16_ids)

    precision_comparison_records.append({
        "sample_id": sample_id,
        "trainer_vs_manual": trainer_ids == manual_ids,
        "trainer_vs_fp32": trainer_ids == fp32_ids,
        "trainer_vs_bf16": trainer_ids == bf16_ids,
        "manual_vs_fp32": manual_ids == fp32_ids,
        "manual_vs_bf16": manual_ids == bf16_ids,
        "fp32_vs_bf16": fp32_ids == bf16_ids
    })

precision_comparison_df = pd.DataFrame(
    precision_comparison_records
)

precision_match_summary = (
    precision_comparison_df
    .drop(columns="sample_id")
    .sum()
    .to_frame("matching_samples")
)

precision_match_summary["match_rate"] = (
    precision_match_summary["matching_samples"]
    / len(precision_comparison_df)
    * 100
)

display(precision_comparison_df)
display(precision_match_summary)

,sample_id,trainer_vs_manual,trainer_vs_fp32,trainer_vs_bf16,manual_vs_fp32,manual_vs_bf16,fp32_vs_bf16
0,7,True,True,True,True,True,True
1,8,True,True,True,True,True,True
2,12,True,True,True,True,True,True
3,17,True,True,True,True,True,True
4,36,True,True,True,True,True,True
5,45,False,True,False,False,True,False
6,47,True,True,True,True,True,True
7,49,False,True,False,False,True,False
8,54,True,True,True,True,True,True
9,61,True,True,True,True,True,True


,matching_samples,match_rate
trainer_vs_manual,17,85.0
trainer_vs_fp32,20,100.0
trainer_vs_bf16,17,85.0
manual_vs_fp32,17,85.0
manual_vs_bf16,20,100.0
fp32_vs_bf16,17,85.0


In [86]:
from unittest.mock import patch

generation_precision_records = []

original_generate = repro_trainer.model.generate

def tracked_generate(*args, **kwargs):

    generation_precision_records.append({
        "autocast_enabled": torch.is_autocast_enabled("cuda"),
        "autocast_dtype": str(torch.get_autocast_dtype("cuda")),
        "model_dtype": str(
            next(repro_trainer.model.parameters()).dtype
        ),
        "training_mode": repro_trainer.model.training
    })

    return original_generate(*args, **kwargs)

with patch.object(
    repro_trainer.model,
    "generate",
    new=tracked_generate
):

    precision_verification_result = repro_trainer.predict(
        diagnostic_dataset
    )

generation_precision_df = pd.DataFrame(
    generation_precision_records
)

display(generation_precision_df)

,autocast_enabled,autocast_dtype,model_dtype,training_mode
0,False,torch.float16,torch.float32,False
1,False,torch.float16,torch.float32,False
2,False,torch.float16,torch.float32,False
3,False,torch.float16,torch.float32,False
4,False,torch.float16,torch.float32,False


In [87]:
from types import SimpleNamespace

def calculate_reproducibility_rouge(generated_rows):

    normalized_rows = [
        normalize_generated_ids(row)
        for row in generated_rows
    ]

    predictions = np.full(
        (len(normalized_rows), GENERATION_MAX_LENGTH),
        tokenizer.pad_token_id,
        dtype=np.int64
    )

    for i, row in enumerate(normalized_rows):

        predictions[i, :len(row)] = row

    eval_pred = SimpleNamespace(
        predictions=predictions,
        label_ids=trainer_prediction_results.label_ids
    )

    random_state = np.random.get_state()

    try:

        np.random.seed(SEED)

        metrics = compute_rouge_metrics(eval_pred)

    finally:

        np.random.set_state(random_state)

    return metrics

In [88]:
rouge_precision_records = []

for method, generated_rows in [
    ("trainer", trainer_generated_ids),
    ("manual", manual_generated_ids),
    ("fp32", fp32_generated_ids),
    ("bf16", bf16_generated_ids)
]:

    metrics = calculate_reproducibility_rouge(
        generated_rows
    )

    rouge_precision_records.append({
        "method": method,
        **metrics
    })

rouge_precision_df = pd.DataFrame(
    rouge_precision_records
)

display(rouge_precision_df)

,method,rouge1,rouge2,rougeL,invalid_generation_count
0,trainer,14.778891,1.685329,13.751730,0
1,manual,14.197240,1.478052,13.274526,0
2,fp32,14.778891,1.685329,13.751730,0
3,bf16,14.197240,1.478052,13.274526,0


In [89]:
rouge_precision_index = rouge_precision_df.set_index(
    "method"
)

rouge_columns = ["rouge1", "rouge2", "rougeL"]

trainer_fp32_rouge_match = np.allclose(
    rouge_precision_index.loc["trainer", rouge_columns]
    .to_numpy(dtype=float),
    rouge_precision_index.loc["fp32", rouge_columns]
    .to_numpy(dtype=float),
    rtol=0,
    atol=1e-10
)

manual_bf16_rouge_match = np.allclose(
    rouge_precision_index.loc["manual", rouge_columns]
    .to_numpy(dtype=float),
    rouge_precision_index.loc["bf16", rouge_columns]
    .to_numpy(dtype=float),
    rtol=0,
    atol=1e-10
)

print("Trainer vs FP32 ROUGE Match:", trainer_fp32_rouge_match)

print("Manual vs BF16 ROUGE Match:", manual_bf16_rouge_match)

Trainer vs FP32 ROUGE Match: True
Manual vs BF16 ROUGE Match: True


In [90]:
with torch.inference_mode():

    with torch.autocast("cuda", enabled=False):

        generated = model.generate(
            **inputs,
            **decoding_configs["A"]
        )

In [91]:
rouge_reproducibility_df = token_comparison_df.merge(
    precision_comparison_df,
    on="sample_id",
    validate="one_to_one"
)

rouge_reproducibility_df["input_ids_match"] = input_ids_match
rouge_reproducibility_df["attention_masks_match"] = attention_masks_match

rouge_reproducibility_df["reference_decoding_match"] = (
    reference_decoding_match
)

rouge_reproducibility_df["generation_config_match"] = (
    generation_config_match
)

rouge_reproducibility_df["trainer_eval_mode"] = trainer_eval_mode

rouge_reproducibility_df["trainer_bf16_config"] = (
    repro_trainer.args.bf16
)

rouge_reproducibility_df["trainer_generation_autocast"] = str(
    generation_precision_df["autocast_enabled"].unique().tolist()
)

rouge_reproducibility_df["trainer_fp32_rouge_match"] = (
    trainer_fp32_rouge_match
)

rouge_reproducibility_df["manual_bf16_rouge_match"] = (
    manual_bf16_rouge_match
)

for method in ["trainer", "manual", "fp32", "bf16"]:

    rouge_reproducibility_df[f"{method}_rougeL"] = (
        rouge_precision_index.loc[method, "rougeL"]
    )

rouge_reproducibility_df["historical_trainer_rougeL"] = (
    13.751729578646366
)

rouge_reproducibility_df["historical_manual_rougeL"] = (
    13.274526093657549
)

rouge_reproducibility_df["historical_rougeL_difference"] = (
    rouge_reproducibility_df["historical_trainer_rougeL"]
    - rouge_reproducibility_df["historical_manual_rougeL"]
)

rouge_reproducibility_df["likely_root_cause"] = (
    "Generation precision mismatch"
)

rouge_reproducibility_df.to_csv(
    reports_dir / "rouge_reproducibility_analysis.csv",
    index=False
)

print("Total Records:", len(rouge_reproducibility_df))
print("Report saved successfully.")

display(rouge_reproducibility_df)

Total Records: 20
Report saved successfully.


,sample_id,trainer_length,manual_length,tokens_match,first_difference_position,trainer_different_token,manual_different_token,trainer_vs_manual,trainer_vs_fp32,trainer_vs_bf16,...,trainer_fp32_rouge_match,manual_bf16_rouge_match,trainer_rougeL,manual_rougeL,fp32_rougeL,bf16_rougeL,historical_trainer_rougeL,historical_manual_rougeL,historical_rougeL_difference,likely_root_cause
0,7,768,768,True,NaN,NaN,NaN,True,True,True,...,True,True,13.75173,13.274526,13.75173,13.274526,13.75173,13.274526,0.477203,Generation precision mismatch
1,8,768,768,True,NaN,NaN,NaN,True,True,True,...,True,True,13.75173,13.274526,13.75173,13.274526,13.75173,13.274526,0.477203,Generation precision mismatch
2,12,747,747,True,NaN,NaN,NaN,True,True,True,...,True,True,13.75173,13.274526,13.75173,13.274526,13.75173,13.274526,0.477203,Generation precision mismatch
3,17,768,768,True,NaN,NaN,NaN,True,True,True,...,True,True,13.75173,13.274526,13.75173,13.274526,13.75173,13.274526,0.477203,Generation precision mismatch
4,36,768,768,True,NaN,NaN,NaN,True,True,True,...,True,True,13.75173,13.274526,13.75173,13.274526,13.75173,13.274526,0.477203,Generation precision mismatch
5,45,768,768,False,2.0,3.0,454.0,False,True,False,...,True,True,13.75173,13.274526,13.75173,13.274526,13.75173,13.274526,0.477203,Generation precision mismatch
6,47,768,768,True,NaN,NaN,NaN,True,True,True,...,True,True,13.75173,13.274526,13.75173,13.274526,13.75173,13.274526,0.477203,Generation precision mismatch
7,49,768,768,False,5.0,8.0,3.0,False,True,False,...,True,True,13.75173,13.274526,13.75173,13.274526,13.75173,13.274526,0.477203,Generation precision mismatch
8,54,768,768,True,NaN,NaN,NaN,True,True,True,...,True,True,13.75173,13.274526,13.75173,13.274526,13.75173,13.274526,0.477203,Generation precision mismatch
9,61,768,768,True,NaN,NaN,NaN,True,True,True,...,True,True,13.75173,13.274526,13.75173,13.274526,13.75173,13.274526,0.477203,Generation precision mismatch


In [92]:
SELECTED_CONFIG = "C"
SELECTED_MODEL = "lora"
SELECTED_CHECKPOINT = "checkpoint-96"
SELECTED_PRECISION = "bf16_autocast"

selected_generation_config = decoding_configs[
    SELECTED_CONFIG
].copy()

display(pd.DataFrame(
    selected_generation_config.items(),
    columns=["parameter", "value"]
))

print("Selected Model:", SELECTED_MODEL)
print("Selected Checkpoint:", SELECTED_CHECKPOINT)
print("Selected Config:", SELECTED_CONFIG)
print("Selected Precision:", SELECTED_PRECISION)

,parameter,value
0,max_length,256
1,num_beams,1
2,repetition_penalty,1.0
3,no_repeat_ngram_size,3
4,do_sample,False


Selected Model: lora
Selected Checkpoint: checkpoint-96
Selected Config: C
Selected Precision: bf16_autocast


In [93]:
generation_quality_comparison["selected_for_final_evaluation"] = (
    (generation_quality_comparison["model"] == SELECTED_MODEL)
    & (generation_quality_comparison["config"] == SELECTED_CONFIG)
)

selection_reasons = {
    ("zero_shot", "A"): (
        "Higher supported claim rate but insufficient content "
        "coverage and severe repetition in one sample."
    ),
    ("zero_shot", "C"): (
        "No trigram repetition but excessively short summaries "
        "and insufficient key finding coverage."
    ),
    ("lora", "A"): (
        "Severe repetition, poor EOS termination and "
        "limited assessable scientific claims."
    ),
    ("lora", "C"): (
        "Selected as development candidate due to zero trigram "
        "repetition, improved EOS termination, competitive ROUGE "
        "and lower generated token count. Scientific reliability "
        "remains insufficient."
    )
}

generation_quality_comparison["selection_reason"] = [
    selection_reasons[(model_name, config_name)]
    for model_name, config_name in zip(
        generation_quality_comparison["model"],
        generation_quality_comparison["config"]
    )
]

generation_quality_comparison["evaluation_scope"] = (
    "development_validation"
)

generation_quality_comparison["inference_precision"] = (
    SELECTED_PRECISION
)

generation_quality_comparison["checkpoint"] = (
    generation_quality_comparison["model"].map({
        "lora": SELECTED_CHECKPOINT,
        "zero_shot": "pretrained_base_model"
    })
)

generation_quality_comparison["max_length"] = (
    generation_quality_comparison["config"].map(
        lambda config: decoding_configs[config]["max_length"]
    )
)

generation_quality_comparison["no_repeat_ngram_size"] = (
    generation_quality_comparison["config"].map(
        lambda config: decoding_configs[config]["no_repeat_ngram_size"]
    )
)

assert (
    generation_quality_comparison[
        "selected_for_final_evaluation"
    ].sum() == 1
)

display(generation_quality_comparison)

,model,config,automatic_sample_count,rouge1,rouge2,rougeL,distinct_2,repeated_trigram_ratio,eos_before_limit_rate,supported_claim_count,...,key_findings_covered,key_findings_total,key_finding_coverage_rate,selected_for_final_evaluation,selection_reason,evaluation_scope,inference_precision,checkpoint,max_length,no_repeat_ngram_size
0,lora,A,20,14.197240,1.478052,13.274526,0.040897,0.956754,10.0,1,...,1,15,6.666667,False,"Severe repetition, poor EOS termination and li...",development_validation,bf16_autocast,checkpoint-96,768,0
1,lora,C,20,26.044892,5.502064,14.671474,0.760291,0.000000,75.0,6,...,1,15,6.666667,True,Selected as development candidate due to zero ...,development_validation,bf16_autocast,checkpoint-96,256,3
2,zero_shot,A,20,5.163355,1.379274,4.262540,0.931898,0.057902,95.0,4,...,1,15,6.666667,False,Higher supported claim rate but insufficient c...,development_validation,bf16_autocast,pretrained_base_model,768,0
3,zero_shot,C,20,5.140493,1.484483,4.085911,0.981943,0.000000,100.0,4,...,1,15,6.666667,False,No trigram repetition but excessively short su...,development_validation,bf16_autocast,pretrained_base_model,256,3


In [94]:
generation_quality_comparison.to_csv(
    reports_dir / "generation_quality_comparison.csv",
    index=False
)

display(generation_quality_comparison[[
    "model",
    "config",
    "rougeL",
    "repeated_trigram_ratio",
    "supported_claim_rate",
    "contradiction_count",
    "key_finding_coverage_rate",
    "selected_for_final_evaluation"
]])

print("Total Records:", len(generation_quality_comparison))
print("Selected Candidates:", generation_quality_comparison[
    "selected_for_final_evaluation"
].sum())

print("Report saved successfully.")

,model,config,rougeL,repeated_trigram_ratio,supported_claim_rate,contradiction_count,key_finding_coverage_rate,selected_for_final_evaluation
0,lora,A,13.274526,0.956754,33.333333,0,6.666667,False
1,lora,C,14.671474,0.000000,31.578947,1,6.666667,True
2,zero_shot,A,4.262540,0.057902,80.000000,1,6.666667,False
3,zero_shot,C,4.085911,0.000000,80.000000,1,6.666667,False


Total Records: 4
Selected Candidates: 1
Report saved successfully.
